In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
import pickle as pkl

In [ ]:
comp = pkl.load(open('/home/tru_exe/projects/earthquakeDSS/data/compiled_data_class.pkl', 'rb'))
comp.head()


,tweet_id,tweet_text,label
0,'451585608612208640',Guys northem chile really needs a support mess...,sympathy_and_emotional_support
1,'451500842395250688',RT @Euphorian54: #Rehearsal time w/ @debbiegib...,not_related_or_irrelevant
2,'451833189439246336',Happy B-Day! @AllRiseSilver https://t.co/jcfCk...,not_related_or_irrelevant
3,'451330668539031552',RT @kuroab_90: My heart goes out to the victim...,sympathy_and_emotional_support
4,'451489071404040192',"Chile Earthquake: 5 Dead, Several Seriously In...",injured_or_dead_people


In [ ]:
comp['tweet_id'] = comp['tweet_id'].str.replace("'" , "").astype(int)

1- Injured or dead people---Reports of casualties and/or injured people due to the crisis

2- Missing, trapped, or found people---Reports and/or questions about missing or found people

3- Displaced people and evacuations---People who have relocated due to the crisis, even for a short time (includes evacuations)

4- Infrastructure and utilities damage---Reports of damaged buildings, roads, bridges, or utilities/services interrupted or restored

5- Donation needs or offers or volunteering services---Reports of urgent needs or donations of shelter and/or supplies such as food, water, clothing, money, medical supplies or blood; and volunteering services

6- Caution and advice---Reports of warnings issued or lifted, guidance and tips

7- Sympathy and emotional support---Prayers, thoughts, and emotional support

8- Other useful information---Other useful information that helps understand the situation

9- Not related or irrelevant---Unrelated to the situation or irrelevant




In [ ]:
severity_map = {
    "injured_or_dead_people":4,
    "missing_trapped_or_found_people":4,
    "displaced_people_and_evacuations":3,
    "infrastructure_and_utilities_damage":3,
    "donation_needs_or_offers_or_volunteering_services":2,
    "caution_and_advice":2,
    "sympathy_and_emotional_support":1,
    "other_useful_information":1,
    "not_related_or_irrelevant":0
}

comp['severity_class'] = comp['label'].map(severity_map)

In [ ]:
comp.head()

,tweet_id,tweet_text,label,severity_class
0,451585608612208640,Guys northem chile really needs a support mess...,sympathy_and_emotional_support,1
1,451500842395250688,RT @Euphorian54: #Rehearsal time w/ @debbiegib...,not_related_or_irrelevant,0
2,451833189439246336,Happy B-Day! @AllRiseSilver https://t.co/jcfCk...,not_related_or_irrelevant,0
3,451330668539031552,RT @kuroab_90: My heart goes out to the victim...,sympathy_and_emotional_support,1
4,451489071404040192,"Chile Earthquake: 5 Dead, Several Seriously In...",injured_or_dead_people,4


In [ ]:
pkl.dump(comp, open('/home/tru_exe/projects/earthquakeDSS/data/compiled_data_w_severityClass.pkl', 'wb'))


In [ ]:
comp = pkl.load(open('/home/tru_exe/projects/earthquakeDSS/data/compiled_data_w_severityClass.pkl', 'rb'))

In [ ]:
comp.head()

,tweet_id,tweet_text,label,severity_class
0,451585608612208640,Guys northem chile really needs a support mess...,sympathy_and_emotional_support,1
1,451500842395250688,RT @Euphorian54: #Rehearsal time w/ @debbiegib...,not_related_or_irrelevant,0
2,451833189439246336,Happy B-Day! @AllRiseSilver https://t.co/jcfCk...,not_related_or_irrelevant,0
3,451330668539031552,RT @kuroab_90: My heart goes out to the victim...,sympathy_and_emotional_support,1
4,451489071404040192,"Chile Earthquake: 5 Dead, Several Seriously In...",injured_or_dead_people,4


In [ ]:
import re

def clean_tweet(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)          # remove URLs
    text = re.sub(r"@\w+", "", text)             # remove mentions
    text = re.sub(r"#(\w+)", r"\1", text)        # keep hashtag text, remove #
    text = re.sub(r"rt\s+", "", text)            # remove RT prefix
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text) # remove special chars
    text = re.sub(r"\s+", " ", text).strip()
    return text
comp["clean_text"] = comp["tweet_text"].apply(clean_tweet)
comp.head()


,tweet_id,tweet_text,label,severity_class,clean_text
0,451585608612208640,Guys northem chile really needs a support mess...,sympathy_and_emotional_support,1,guys northem chile really needs a suppomessage...
1,451500842395250688,RT @Euphorian54: #Rehearsal time w/ @debbiegib...,not_related_or_irrelevant,0,rehearsal time w amp teamdeb for hitparade amp...
2,451833189439246336,Happy B-Day! @AllRiseSilver https://t.co/jcfCk...,not_related_or_irrelevant,0,happy b day happyhyukday
3,451330668539031552,RT @kuroab_90: My heart goes out to the victim...,sympathy_and_emotional_support,1,my heagoes out to the victims of the 8 2 magni...
4,451489071404040192,"Chile Earthquake: 5 Dead, Several Seriously In...",injured_or_dead_people,4,chile earthquake 5 dead several seriously injured


In [ ]:
import tensorflow as tf

In [ ]:
from gensim.models import Word2Vec
import gensim
from nltk.tokenize import word_tokenize, sent_tokenize
import nltk
import warnings

In [ ]:
from gensim.models import KeyedVectors

# Path to your downloaded CrisisNLP .bin file
CRISIS_BIN_PATH = "/home/tru_exe/projects/earthquakeDSS/data/crisisNLP_word2vec_model/crisisNLP_word_vector.bin"   
print("Loading CrisisNLP embeddings...")
crisis_wv = KeyedVectors.load_word2vec_format(CRISIS_BIN_PATH, binary=True)

EMBED_DIM = crisis_wv.vector_size  
print(f"Loaded {len(crisis_wv)} vectors | Dimension: {EMBED_DIM}")

Loading CrisisNLP embeddings...
Loaded 2152854 vectors | Dimension: 300


In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

MAX_VOCAB = 60000
MAX_LEN   = 100

tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token="<OOV>")
tokenizer.fit_on_texts(comp["clean_text"])

word_index = tokenizer.word_index

sequences = tokenizer.texts_to_sequences(comp["clean_text"])
X = pad_sequences(sequences, maxlen=MAX_LEN, padding="post", truncating="post")
y = comp["severity_class"].values

In [ ]:
pkl.dump((X, y), open('/home/tru_exe/projects/earthquakeDSS/data/X_y_sequences.pkl', 'wb'))

In [18]:
X[10]

array([ 201,   12,  812,    4, 1061,    7,    2,   35,   44, 1629,    7,
        102, 1063,   12,   45,  129,   29,   53,   79,    3,    6,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0], dtype=int32)

In [19]:
print(word_index)
print(X)

{'<OOV>': 1, 'chile': 2, 'earthquake': 3, 'the': 4, 'in': 5, 'tsunami': 6, 'of': 7, '8': 8, 'prayforchile': 9, 'a': 10, 'to': 11, 'and': 12, '2': 13, 'for': 14, 's': 15, 'magnitude': 16, 'i': 17, 'after': 18, 'you': 19, 'quake': 20, 'coast': 21, 'is': 22, 'off': 23, '7': 24, 'on': 25, 'my': 26, 'are': 27, 'from': 28, 'all': 29, 'at': 30, 'northern': 31, 'that': 32, 'by': 33, 'warning': 34, 'with': 35, 'people': 36, 'iquique': 37, '6': 38, 'amp': 39, '4': 40, '5': 41, 'hit': 42, '3': 43, 'an': 44, 'prayers': 45, 'this': 46, 'news': 47, 'we': 48, 'chileearthquake': 49, 'it': 50, 'massive': 51, 'peru': 52, 'be': 53, 'just': 54, 'm': 55, 'please': 56, 'pray': 57, 'as': 58, 'was': 59, 't': 60, 'has': 61, 'powerful': 62, 'so': 63, 'now': 64, 'there': 65, 'god': 66, 'strikes': 67, '2014': 68, 'hits': 69, 'out': 70, 'me': 71, 'aftershock': 72, '0': 73, '1': 74, 'have': 75, 'following': 76, 'hope': 77, 'not': 78, 'safe': 79, 'breaking': 80, 'our': 81, 'about': 82, 'dead': 83, 'another': 84, 'ch

In [23]:
vocab_size = min(MAX_VOCAB, len(word_index)) + 1
embedding_matrix = np.zeros((vocab_size, EMBED_DIM), dtype="float32")

hits, misses = 0, 0
for word, idx in word_index.items():
    if idx >= vocab_size:
        continue
    if word in crisis_wv:
        embedding_matrix[idx] = crisis_wv[word]
        hits += 1
    else:
        # Random init with same scale as Word2Vec vectors
        embedding_matrix[idx] = np.random.uniform(-0.25, 0.25, EMBED_DIM)
        misses += 1
        
print(f"Vocab coverage — Hits: {hits} | OOV: {misses} ({misses/(hits+misses)*100:.1f}%)")
print(f"Embedding matrix: {embedding_matrix.shape}")

Vocab coverage — Hits: 3517 | OOV: 670 (16.0%)
Embedding matrix: (4188, 300)


In [24]:
embedding_matrix[:10]  # Show first 5 vectors (including OOV token)


array([[ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [ 0.04093697,  0.09200399,  0.01902585, ...,  0.221283  ,
         0.05872974, -0.01734496],
       [ 0.343046  , -0.274161  , -0.155809  , ..., -0.452724  ,
         0.380765  , -0.148149  ],
       ...,
       [ 0.23473755,  0.04530949,  0.2323674 , ...,  0.20705217,
        -0.20676133, -0.11118555],
       [ 0.10430983,  0.224283  ,  0.23448117, ..., -0.16489221,
         0.13336535,  0.08230148],
       [ 0.203444  , -0.164255  , -0.070287  , ..., -0.062831  ,
        -0.288942  , -0.184035  ]], shape=(10, 300), dtype=float32)

In [25]:
pkl.dump((embedding_matrix), open('/home/tru_exe/projects/earthquakeDSS/data/embedding_matrix.pkl', 'wb'))

In [26]:
import pickle as pkl
emb = pkl.load(open('/home/tru_exe/projects/earthquakeDSS/data/embedding_matrix.pkl', 'rb'))
emb.shape

(4188, 300)

In [27]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout

In [28]:
def build_model(vocab_size, embed_dim, embedding_matrix, max_len , num_classes=5):
    
    model = Sequential([
        Embedding(
            input_dim = vocab_size,
            output_dim = embed_dim,
            weights = [embedding_matrix],
            trainable = False,
            name="embedding_layer"
        ),
        
        Bidirectional(LSTM(256, return_sequences=True) , name = "bilstm1"),
        Dropout(0.5),
        Bidirectional(LSTM(128 , return_sequences=False)),
        Dropout(0.5),
        Dense(64, activation="relu" , name="dense1"),
        Dense(num_classes, activation="softmax" , name="output")        
    ])
    
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

In [ ]:
import tensorflow as tf


[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]
GPU devices: []


ValueError: No matching devices found for 'GPU:0'

In [32]:

print("All devices:", tf.config.list_physical_devices())

All devices: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


In [ ]:
model = build_model(vocab_size = vocab_size, embed_dim = EMBED_DIM, embedding_matrix = embedding_matrix, max_len = MAX_LEN , num_classes=5)
model.summary()

NameError: name 'vocab_size' is not defined

In [24]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

/home/tru_exe/projects/earthquakeDSS/ml/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [26]:
tokenizer = AutoTokenizer.from_pretrained(
    "distilbert/distilbert-base-uncased",
)

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert/distilbert-base-uncased",
    num_labels=4,
    problem_type="multi_label_classification"
    
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1610.07it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
